# OPUS data extraction (ATR-FTIR, *Pinus pinaster*, *Fusarium circinatum*)

Reads the raw Bruker OPUS files (`.0`) for timepoints T0 to T7 and writes the CSV files used by the other notebooks. Everything goes to `Outputs/Spectra_Data`. `Combined_spectra_data.csv` holds wavenumber and absorbance for every sample. `Data_infection_status.csv` adds the metadata parsed from the sample name. `Full_data_infection_status.csv` puts all samples on the same wavenumber axis and is the input for the rest of the analysis.

The last cells check the number of points per sample and the number of samples per timepoint, trial and infection status.

## Imports and paths

`opusFC` reads the Bruker files. `OUTPUT_DIR` is created if it does not exist.

In [1]:
%%time
import os
import numpy as np
import pandas as pd
import opusFC
%matplotlib inline

BASE_DIR = r'C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster'
OUTPUT_DIR = os.path.join(BASE_DIR, 'Outputs', 'Spectra_Data')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CPU times: total: 1.48 s
Wall time: 1.79 s


## Extract spectra

Reads every `.0` file in `Raw_Data/T0` to `Raw_Data/T7`. The absorbance block is looked up by the usual names (`AB`, `Absorbance`, `ABDATA`, `ABSG`); if none is present, the first block in the file is used. Files that cannot be read are skipped.

In [2]:
%%time
ABSORBANCE_BLOCKS = ['AB', 'Absorbance', 'ABDATA', 'ABSG']


def read_opus_file(file_path):
    try:
        contents = opusFC.listContents(file_path)
        if not contents:
            return None
        for block in ABSORBANCE_BLOCKS:
            if block in contents:
                return opusFC.getOpusData(file_path, block)
        return opusFC.getOpusData(file_path, contents[0])
    except Exception:
        return None


def extract_all_spectra(folder_paths, output_csv):
    all_data = []
    for folder_path in folder_paths:
        if not os.path.exists(folder_path):
            continue
        count = 0
        for file in os.listdir(folder_path):
            full_path = os.path.join(folder_path, file)
            if not (os.path.isfile(full_path) and file.lower().endswith('.0')):
                continue
            ab_data = read_opus_file(full_path)
            if ab_data is not None:
                all_data.append(pd.DataFrame({
                    'wavenumber': ab_data.x,
                    'absorbance': ab_data.y,
                    'sample_name': os.path.splitext(file)[0],
                }))
                count += 1
        print(f"Processed {count} files from {folder_path}")
    if not all_data:
        return None
    combined_df = pd.concat(all_data, ignore_index=True)
    combined_df.to_csv(output_csv, index=False)
    return combined_df


FOLDERS = [os.path.join(BASE_DIR, 'Raw_Data', f'T{i}') for i in range(8)]

file_path = os.path.join(OUTPUT_DIR, 'Combined_spectra_data.csv')
combined_data = extract_all_spectra(FOLDERS, file_path)

Processed 380 files from C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster\Raw_Data\T0
Processed 382 files from C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster\Raw_Data\T1
Processed 382 files from C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster\Raw_Data\T2
Processed 382 files from C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster\Raw_Data\T3
Processed 382 files from C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster\Raw_Data\T4
Processed 382 files from C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster\Raw_Data\T5
Processed 382 files from C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster\Raw_Data\T6
Processed 382 files from C:/Users/MCL/Desktop/ML_ATR_FTIR_Spectra_Ppinaster\Raw_Data\T7
CPU times: total: 33.6 s
Wall time: 34 s


## Load combined spectra

Reads the CSV back from disk.

In [3]:
%%time
file_path = os.path.join(OUTPUT_DIR, 'Combined_spectra_data.csv')
df = pd.read_csv(file_path)
df

CPU times: total: 3.47 s
Wall time: 3.48 s


,wavenumber,absorbance,sample_name
0,3996.672393,0.009109,Pp_E1_T0_CT_01_1
1,3994.613314,0.009036,Pp_E1_T0_CT_01_1
2,3992.554235,0.008813,Pp_E1_T0_CT_01_1
3,3990.495156,0.008795,Pp_E1_T0_CT_01_1
4,3988.436077,0.008893,Pp_E1_T0_CT_01_1
...,...,...,...
5338387,407.694596,0.438713,Pp_E3_T7_FC_30_2
5338388,405.635532,0.441002,Pp_E3_T7_FC_30_2
5338389,403.576469,0.438678,Pp_E3_T7_FC_30_2
5338390,401.517405,0.436239,Pp_E3_T7_FC_30_2


## Unique wavenumbers

Number of distinct wavenumbers and the first 20 values.

In [4]:
%%time
unique_wavenumbers = df['wavenumber'].unique()
print(f"Number of unique wavenumbers: {len(unique_wavenumbers)}")
print("First 20 unique wavenumbers:", unique_wavenumbers[:20])

Number of unique wavenumbers: 8740
First 20 unique wavenumbers: [3996.6723931  3994.61331407 3992.55423504 3990.49515601 3988.43607699
 3986.37699796 3984.31791893 3982.2588399  3980.19976087 3978.14068185
 3976.08160282 3974.02252379 3971.96344476 3969.90436574 3967.84528671
 3965.78620768 3963.72712865 3961.66804962 3959.6089706  3957.54989157]
CPU times: total: 93.8 ms
Wall time: 105 ms


## Sample metadata

Infection status (`FC` in the name means Fusarium, otherwise control), trial (E1, E2 or E3), timepoint, sample number and replicate are taken from the sample name.

In [5]:
%%time
def get_trial(name):
    for trial in ('E1', 'E2', 'E3'):
        if trial in name:
            return trial
    return 'UNKNOWN'


is_fusarium = df['sample_name'].str.contains('FC')

df['infection_status'] = np.where(is_fusarium, 'FUSARIUM', 'CONTROL')
df['trial'] = df['sample_name'].apply(get_trial)
df['status'] = np.where(is_fusarium, 'FC', 'CT')
df['timepoint'] = df['sample_name'].str.extract(r'(T\d+)')
df[['sample_num', 'replicate']] = df['sample_name'].str.extract(r'_(\d+)_(\d+)$').astype(str)

df.to_csv(os.path.join(OUTPUT_DIR, 'Data_infection_status.csv'), index=False)
df

CPU times: total: 1min 4s
Wall time: 1min 5s


,wavenumber,absorbance,sample_name,infection_status,trial,status,timepoint,sample_num,replicate
0,3996.672393,0.009109,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,01,1
1,3994.613314,0.009036,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,01,1
2,3992.554235,0.008813,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,01,1
3,3990.495156,0.008795,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,01,1
4,3988.436077,0.008893,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,01,1
...,...,...,...,...,...,...,...,...,...
5338387,407.694596,0.438713,Pp_E3_T7_FC_30_2,FUSARIUM,E3,FC,T7,30,2
5338388,405.635532,0.441002,Pp_E3_T7_FC_30_2,FUSARIUM,E3,FC,T7,30,2
5338389,403.576469,0.438678,Pp_E3_T7_FC_30_2,FUSARIUM,E3,FC,T7,30,2
5338390,401.517405,0.436239,Pp_E3_T7_FC_30_2,FUSARIUM,E3,FC,T7,30,2


## Common wavenumber axis

Every sample gets the wavenumbers of the first sample, rounded to 2 decimals, which removes small drift between spectra. This assumes all samples have the same number of points in the same order; the next cells check it.

In [6]:
%%time
file_path = os.path.join(OUTPUT_DIR, 'Data_infection_status.csv')
df = pd.read_csv(file_path, low_memory=False)

first_sample = df['sample_name'].iloc[0]
reference_wavenumbers = df.loc[df['sample_name'] == first_sample, 'wavenumber'].values
n_repeats = len(df) // len(reference_wavenumbers) + 1
df['wavenumber'] = np.tile(reference_wavenumbers, n_repeats)[:len(df)]
df['wavenumber'] = df['wavenumber'].round(2)

df.to_csv(os.path.join(OUTPUT_DIR, 'Full_data_infection_status.csv'), index=False)
df

CPU times: total: 41.7 s
Wall time: 42.5 s


,wavenumber,absorbance,sample_name,infection_status,trial,status,timepoint,sample_num,replicate
0,3996.67,0.009109,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,1,1
1,3994.61,0.009036,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,1,1
2,3992.55,0.008813,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,1,1
3,3990.50,0.008795,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,1,1
4,3988.44,0.008893,Pp_E1_T0_CT_01_1,CONTROL,E1,CT,T0,1,1
...,...,...,...,...,...,...,...,...,...
5338387,407.70,0.438713,Pp_E3_T7_FC_30_2,FUSARIUM,E3,FC,T7,30,2
5338388,405.64,0.441002,Pp_E3_T7_FC_30_2,FUSARIUM,E3,FC,T7,30,2
5338389,403.58,0.438678,Pp_E3_T7_FC_30_2,FUSARIUM,E3,FC,T7,30,2
5338390,401.52,0.436239,Pp_E3_T7_FC_30_2,FUSARIUM,E3,FC,T7,30,2


## Unique wavenumbers after alignment

Same check as before, on the aligned data.

In [7]:
%%time
unique_wavenumbers = df['wavenumber'].unique()
print(f"Number of unique wavenumbers: {len(unique_wavenumbers)}")
print("First 20 unique wavenumbers:", unique_wavenumbers[:20])

Number of unique wavenumbers: 1748
First 20 unique wavenumbers: [3996.67 3994.61 3992.55 3990.5  3988.44 3986.38 3984.32 3982.26 3980.2
 3978.14 3976.08 3974.02 3971.96 3969.9  3967.85 3965.79 3963.73 3961.67
 3959.61 3957.55]
CPU times: total: 141 ms
Wall time: 129 ms


## Points per sample

Every sample should have the same count.

In [8]:
%%time
counts_per_sample = df.groupby('sample_name')['wavenumber'].count()
print(counts_per_sample)

sample_name
Pp_E1_T0_CT_01_1    1748
Pp_E1_T0_CT_01_2    1748
Pp_E1_T0_CT_02_1    1748
Pp_E1_T0_CT_02_2    1748
Pp_E1_T0_CT_03_1    1748
                    ... 
Pp_E3_T7_FC_28_2    1748
Pp_E3_T7_FC_29_1    1748
Pp_E3_T7_FC_29_2    1748
Pp_E3_T7_FC_30_1    1748
Pp_E3_T7_FC_30_2    1748
Name: wavenumber, Length: 3054, dtype: int64
CPU times: total: 625 ms
Wall time: 629 ms


## Samples per timepoint, trial and infection status

Number of unique samples in the final dataset.

In [9]:
%%time
file_path = os.path.join(OUTPUT_DIR, 'Full_data_infection_status.csv')
df = pd.read_csv(file_path)

result = (
    df.groupby(['timepoint', 'trial', 'infection_status'])['sample_num']
    .nunique()
    .reset_index(name='unique_sample_count')
)
print(result)

   timepoint trial infection_status  unique_sample_count
0         T0    E1          CONTROL                   32
1         T0    E1         FUSARIUM                   32
2         T0    E2          CONTROL                   32
3         T0    E2         FUSARIUM                   32
4         T0    E3          CONTROL                   32
5         T0    E3         FUSARIUM                   30
6         T1    E1          CONTROL                   32
7         T1    E1         FUSARIUM                   32
8         T1    E2          CONTROL                   33
9         T1    E2         FUSARIUM                   34
10        T1    E3          CONTROL                   30
11        T1    E3         FUSARIUM                   30
12        T2    E1          CONTROL                   32
13        T2    E1         FUSARIUM                   32
14        T2    E2          CONTROL                   33
15        T2    E2         FUSARIUM                   34
16        T2    E3          CON